## Creating a FunctionTool

In [1]:
from llama_index.core.tools import FunctionTool

def get_weather(location: str) -> str:
    """Useful for getting the weather for a given location."""
    print(f"Getting weather for {location}")
    return f"The weather in {location} is sunny"

tool = FunctionTool.from_defaults(
    get_weather,
    name="my_weather_tool",
    description="Useful for getting weather for a given location",
)
tool.call("New York")

Getting weather for New York


ToolOutput(blocks=[TextBlock(block_type='text', text='The weather in New York is sunny')], tool_name='my_weather_tool', raw_input={'args': ('New York',), 'kwargs': {}}, raw_output='The weather in New York is sunny', is_error=False)

In [2]:
from llama_index.core import VectorStoreIndex
from llama_index.core.tools import QueryEngineTool
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI
from llama_index.embeddings.huggingface import HuggingFaceEmbedding
from llama_index.vector_stores.chroma import ChromaVectorStore
import chromadb

embed_model = HuggingFaceEmbedding("BAAI/bge-small-en-v1.5")

db = chromadb.PersistentClient(path="./alfred_chroma_db")
chroma_collection = db.get_or_create_collection("afred")
vector_store = ChromaVectorStore(chroma_collection=chroma_collection)

index = VectorStoreIndex.from_vector_store(vector_store=vector_store, embed_model=embed_model)

llm = HuggingFaceInferenceAPI(model_name="Qwen/Qwen2.5-Coder-32B-Instruct")
query_engine = index.as_query_engine(llm=llm)
tool = QueryEngineTool.from_defaults(query_engine, name="some useful name", description="some useful description")

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 13029.65it/s]


## Creating Toolspecs

In [3]:
from llama_index.tools.google import GmailToolSpec

tool_spec = GmailToolSpec()
tool_spec_list = tool_spec.to_tool_list()

In [4]:
[(tool.metadata.name, tool.metadata.description) for tool in tool_spec_list]

[('load_data',
  "load_data() -> List[llama_index.core.schema.Document]\nLoad emails from the user's account."),
 ('search_messages',
  "search_messages(query: str, max_results: int | None = None)\n\nSearches email messages given a query string and the maximum number\nof results requested by the user\n   Returns: List of relevant message objects up to the maximum number of results.\n\nArgs:\n    query (str): The user's query\n    max_results (Optional[int]): The maximum number of search results\n    to return."),
 ('create_draft',
  "create_draft(to: List[str] | None = None, subject: str | None = None, message: str | None = None) -> str\n\nCreate and insert a draft email.\n   Print the returned draft's message and id.\n   Returns: Draft object, including draft id and message meta data.\n\nArgs:\n    to (Optional[str]): The email addresses to send the message to\n    subject (Optional[str]): The subject for the event\n    message (Optional[str]): The message for the event"),
 ('update_d

### Model Context Protocol (MCP) in LlamaIndex

In [8]:
from huggingface_hub import login
login()

In [ ]:
import os
from llama_index.tools.mcp import BasicMCPClient, McpToolSpec
from llama_index.core.agent.workflow import ReActAgent
from llama_index.core.workflow import Context
from llama_index.llms.huggingface_api import HuggingFaceInferenceAPI

llm = HuggingFaceInferenceAPI(
    model_name="Qwen/Qwen2.5-Coder-32B-Instruct",
)

mcp_client = BasicMCPClient("https://mcp.deepwiki.com/mcp")
mcp_tool = McpToolSpec(client=mcp_client)

async def get_agent(tool_spec: McpToolSpec):
    tools = await tool_spec.to_tool_list_async()
    return ReActAgent(
        name="DeepWikiAgent",
        description="Answers questions about public GitHub repos",
        tools=tools,
        llm=llm,
        system_prompt="Use the available tools to answer questions about GitHub repositories.",
    )

agent = await get_agent(mcp_tool)
agent_context = Context(agent)

response = await agent.run(
    "Using DeepWiki, what does the run-llama/llama_index repo do?",
    ctx=agent_context,
)
print(response)

WorkflowHandler(workflow=llama_index.core.agent.workflow.react_agent.ReActAgent, run_id=fs3P1Hhqlz, result=None)
